# CUDA OPTIMISED CODE

In [2]:
!pip install numba-cuda[cu12]

In [10]:
from __future__ import print_function
import numpy as np
from numba import cuda
import math

# CUDA device function for cnd
@cuda.jit(device=True)
def cnd_device(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d)) # Use math.fabs for scalar absolute value
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) * # Use math.exp for scalar exponential
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))

    # Replace np.where with an if/else for scalar operation
    if d > 0:
        return 1.0 - ret_val
    else:
        return ret_val

# CUDA kernel for black_scholes
@cuda.jit
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility,
                         callResult, putResult):
    idx = cuda.grid(1) # Get the global thread ID

    # Ensure we don't go out of bounds for the arrays
    if idx < stockPrice.size:
        # Load scalar values for the current option from device memory
        S = stockPrice[idx]
        X = optionStrike[idx]
        T = optionYears[idx]
        R = Riskfree[idx]
        V = Volatility[idx]

        sqrtT = math.sqrt(T) # Use math.sqrt for scalar square root

        # Handle potential division by zero or log of non-positive numbers
        # This is a simplified check; more robust error handling might be needed
        if V * sqrtT == 0 or S / X <= 0:
            callResult[idx] = 0.0
            putResult[idx] = 0.0
            return

        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT) # Use math.log for scalar logarithm
        d2 = d1 - V * sqrtT

        cndd1 = cnd_device(d1) # Call the device function
        cndd2 = cnd_device(d2)

        expRT = math.exp(- R * T) # Use math.exp for scalar exponential

        # Store results back to device memory
        callResult[idx] = S * cndd1 - X * expRT * cndd2
        putResult[idx] = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# Host-side function to orchestrate data transfer and kernel launch
def black_scholes_optimized(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    # Ensure inputs are NumPy arrays and convert to float64 for precision
    stockPrice = np.asarray(stockPrice, dtype=np.float64)
    optionStrike = np.asarray(optionStrike, dtype=np.float64)
    optionYears = np.asarray(optionYears, dtype=np.float64)
    Riskfree = np.asarray(Riskfree, dtype=np.float64)
    Volatility = np.asarray(Volatility, dtype=np.float64)

    N = stockPrice.size

    # Transfer input arrays from host to device
    d_stockPrice = cuda.to_device(stockPrice)
    d_optionStrike = cuda.to_device(optionStrike)
    d_optionYears = cuda.to_device(optionYears)
    d_Riskfree = cuda.to_device(Riskfree)
    d_Volatility = cuda.to_device(Volatility)

    # Allocate output arrays on the device
    d_callResult = cuda.device_array(N, dtype=np.float64)
    d_putResult = cuda.device_array(N, dtype=np.float64)

    # Configure kernel launch parameters
    threadsperblock = 256 # A common choice for threads per block
    blockspergrid = (N + (threadsperblock - 1)) // threadsperblock # Calculate blocks needed

    # Launch the CUDA kernel
    black_scholes_kernel[blockspergrid, threadsperblock](
        d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
        d_callResult, d_putResult
    )

    # Copy results back from device to host
    callResult = d_callResult.copy_to_host()
    putResult = d_putResult.copy_to_host()

    return callResult, putResult

def black_scholes_gpu_resident(d_S, d_X, d_T, d_R, d_V, d_call, d_put, bpg, tpb=256):
    """
    Zero-copy GPU kernel launcher for batch or simulation workloads.
    Operates entirely on pre-transferred device arrays; no host-device
    transfers occur inside this function. Results are written in-place
    to the caller-supplied d_call and d_put device arrays.
    """
    black_scholes_kernel[bpg, tpb](d_S, d_X, d_T, d_R, d_V, d_call, d_put)



# --- Original CPU functions for verification ---
def cnd_cpu(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S = stockPrice
    X = optionStrike
    T = optionYears
    R = Riskfree
    V = Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1)
    cndd2 = cnd_cpu(d2)

    expRT = np.exp(- R * T)

    callResult = S * cndd1 - X * expRT * cndd2
    putResult = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

    return callResult, putResult

def black_scholes_cpu_resident(S, X, T, R, V, call_out, put_out):
    """
    In-place CPU Black-Scholes pricing.
    Accepts pre-allocated call_out and put_out arrays and writes results
    directly into them, avoiding per-call output allocation overhead.
    Direct equivalent of black_scholes_gpu_resident for fair batch comparison.
    """
    sqrtT = np.sqrt(T)
    d1    = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2    = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1)
    cndd2 = cnd_cpu(d2)
    expRT = np.exp(-R * T)
    np.copyto(call_out, S * cndd1 - X * expRT * cndd2)
    np.copyto(put_out,  X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1))



# --- Example Usage and Verification ---
N_OPTIONS = 10**6 # Number of options to price
# Generate random data, ensuring positive values where required to avoid math domain errors
stockPrice = np.random.rand(N_OPTIONS).astype(np.float64) * 100 + 10
optionStrike = np.random.rand(N_OPTIONS).astype(np.float64) * 100 + 10
optionYears = np.random.rand(N_OPTIONS).astype(np.float64) * 2 + 0.1
Riskfree = np.random.rand(N_OPTIONS).astype(np.float64) * 0.1 + 0.01
Volatility = np.random.rand(N_OPTIONS).astype(np.float64) * 0.5 + 0.01

# Run the optimized Numba-CUDA version
call_opt, put_opt = black_scholes_optimized(stockPrice, optionStrike, optionYears, Riskfree, Volatility)

print("Optimized Call Result (first 5):", call_opt[:5])
print("Optimized Put Result (first 5):", put_opt[:5])

# Run the original CPU version for comparison
call_cpu, put_cpu = black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility)

print("\nCPU Call Result (first 5):", call_cpu[:5])
print("CPU Put Result (first 5):", put_cpu[:5])

# Verify that the results are numerically close
np.testing.assert_allclose(call_opt, call_cpu, rtol=1e-5, atol=1e-8)
np.testing.assert_allclose(put_opt, put_cpu, rtol=1e-5, atol=1e-8)
print("\nVerification successful: Results from Numba-CUDA match the original CPU version.")

Optimized Call Result (first 5): [7.35657199e-02 4.38098617e+01 5.74863588e+01 3.96806133e+00
 2.18490892e-55]
Optimized Put Result (first 5): [1.86186196e+01 0.00000000e+00 3.50808787e-15 3.91017032e+01
 8.14298354e+01]

CPU Call Result (first 5): [7.35657199e-02 4.38098617e+01 5.74863588e+01 3.96806133e+00
 2.18490892e-55]
CPU Put Result (first 5): [1.86186196e+01 0.00000000e+00 3.50808787e-15 3.91017032e+01
 8.14298354e+01]

Verification successful: Results from Numba-CUDA match the original CPU version.


# Keeping Data in GPU Memory

In [11]:
import warnings
import time
import timeit
import statistics

import numpy as np
from numba import cuda
from numba.core.errors import NumbaPerformanceWarning
import math

warnings.filterwarnings("ignore", category=NumbaPerformanceWarning)


# ─────────────────────────────────────────────────────────────────────────────
# INPUT FACTORIES
# ─────────────────────────────────────────────────────────────────────────────

def make_inputs(n):
    """Generate pageable host arrays with randomised option parameters."""
    return dict(
        stockPrice   = np.random.uniform(10,   150, n).astype(np.float64),
        optionStrike = np.random.uniform(10,   150, n).astype(np.float64),
        optionYears  = np.random.uniform(0.1,  2.0, n).astype(np.float64),
        Riskfree     = np.random.uniform(0.01, 0.1, n).astype(np.float64),
        Volatility   = np.random.uniform(0.01, 0.5, n).astype(np.float64),
    )


def make_inputs_pinned(n):
    """
    Generate page-locked (pinned) host arrays with randomised option parameters.
    Pinned memory enables direct DMA transfers, eliminating the CPU-side
    staging copy and saturating available PCIe bandwidth.
    """
    def pinned_uniform(lo, hi, size):
        arr = cuda.pinned_array(size, dtype=np.float64)
        arr[:] = np.random.uniform(lo, hi, size)
        return arr

    return dict(
        stockPrice   = pinned_uniform(10,   150, n),
        optionStrike = pinned_uniform(10,   150, n),
        optionYears  = pinned_uniform(0.1,  2.0, n),
        Riskfree     = pinned_uniform(0.01, 0.1, n),
        Volatility   = pinned_uniform(0.01, 0.5, n),
    )


def make_device_inputs(inp):
    """
    Transfer host arrays to the device and pre-allocate output buffers.
    Reusing these buffers across repeated kernel launches avoids
    per-call device allocation overhead.
    """
    N = inp["stockPrice"].size
    return {
        "d_S"    : cuda.to_device(inp["stockPrice"]),
        "d_X"    : cuda.to_device(inp["optionStrike"]),
        "d_T"    : cuda.to_device(inp["optionYears"]),
        "d_R"    : cuda.to_device(inp["Riskfree"]),
        "d_V"    : cuda.to_device(inp["Volatility"]),
        "d_call" : cuda.device_array(N, dtype=np.float64),
        "d_put"  : cuda.device_array(N, dtype=np.float64),
        "bpg"    : (N + THREADSPERBLOCK - 1) // THREADSPERBLOCK,
    }


def make_device_inputs_pinned(inp_pinned):
    """
    Transfer pinned host arrays to the device and pre-allocate output buffers.
    Combining pinned source memory with pre-allocated outputs minimises
    both transfer latency and device-side allocation overhead.
    """
    N = inp_pinned["stockPrice"].size
    return {
        "d_S"    : cuda.to_device(inp_pinned["stockPrice"]),
        "d_X"    : cuda.to_device(inp_pinned["optionStrike"]),
        "d_T"    : cuda.to_device(inp_pinned["optionYears"]),
        "d_R"    : cuda.to_device(inp_pinned["Riskfree"]),
        "d_V"    : cuda.to_device(inp_pinned["Volatility"]),
        "d_call" : cuda.device_array(N, dtype=np.float64),
        "d_put"  : cuda.device_array(N, dtype=np.float64),
        "bpg"    : (N + THREADSPERBLOCK - 1) // THREADSPERBLOCK,
    }


# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

if not cuda.is_available():
    raise RuntimeError("No CUDA-capable GPU detected. This benchmark requires a GPU.")

gpu = cuda.get_current_device()
print(f"GPU                : {gpu.name.decode()}")
print(f"Compute Capability : {gpu.compute_capability}")
print(f"Max Threads/Block  : {gpu.MAX_THREADS_PER_BLOCK}")
print(f"Multiprocessors    : {gpu.MULTIPROCESSOR_COUNT}\n")

REPEATS         = 10
NUMBER          = 5
THREADSPERBLOCK = 256
SIZES           = [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]
BATCH_SIZE      = 100

np.random.seed(42)

# One-time JIT compilation — excluded from all timed measurements
print("Compiling CUDA kernel ...")
black_scholes_optimized(**make_inputs(1024))
cuda.synchronize()
print("Compilation complete.\n")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — End-to-End Latency: Pageable Memory
# Measures the full pipeline (H2D + kernel + D2H) using standard
# pageable host allocations. Establishes the unoptimised baseline.
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("PHASE 1 — End-to-End Latency: Pageable Memory")
print(f"  {REPEATS} repeats × {NUMBER} calls each\n")
print(f"{'Size':>10} {'Min(ms)':>10} {'Max(ms)':>10} {'Mean(ms)':>10} "
      f"{'Std(ms)':>10} {'Tput M/s':>10}")
print("─" * 62)

e2e_results = []
for n in SIZES:
    inp = make_inputs(n)
    black_scholes_optimized(**inp); cuda.synchronize()

    raw = timeit.repeat(
        stmt    = "black_scholes_optimized(**inp); cuda.synchronize()",
        setup   = "pass",
        globals = {"black_scholes_optimized": black_scholes_optimized,
                   "cuda": cuda, "inp": inp},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    per_call_ms = [t / NUMBER * 1000 for t in raw]
    mn = min(per_call_ms); mx = max(per_call_ms)
    mean = statistics.mean(per_call_ms); std = statistics.stdev(per_call_ms)
    tput = n / (mean / 1000) / 1e6
    e2e_results.append(dict(size=n, min=mn, max=mx, mean=mean, std=std, throughput=tput))
    print(f"{n:>10,} {mn:>10.3f} {mx:>10.3f} {mean:>10.3f} {std:>10.4f} {tput:>10.2f}")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1b — End-to-End Latency: Pinned Memory
# Identical pipeline to Phase 1 with page-locked host allocations.
# Direct DMA transfers bypass the CPU staging copy, reducing PCIe latency.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 1b — End-to-End Latency: Pinned Memory")
print(f"  {REPEATS} repeats × {NUMBER} calls each\n")
print(f"{'Size':>10} {'Min(ms)':>10} {'Max(ms)':>10} {'Mean(ms)':>10} "
      f"{'Std(ms)':>10} {'Tput M/s':>10} {'vs 1a':>10}")
print("─" * 74)

e2e_pinned_results = []
for idx, n in enumerate(SIZES):
    inp_p = make_inputs_pinned(n)
    black_scholes_optimized(**inp_p); cuda.synchronize()

    raw = timeit.repeat(
        stmt    = "black_scholes_optimized(**inp_p); cuda.synchronize()",
        setup   = "pass",
        globals = {"black_scholes_optimized": black_scholes_optimized,
                   "cuda": cuda, "inp_p": inp_p},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    per_call_ms = [t / NUMBER * 1000 for t in raw]
    mn = min(per_call_ms); mx = max(per_call_ms)
    mean = statistics.mean(per_call_ms); std = statistics.stdev(per_call_ms)
    tput = n / (mean / 1000) / 1e6
    improvement = e2e_results[idx]["mean"] / mean
    e2e_pinned_results.append(dict(size=n, min=mn, max=mx, mean=mean, std=std, throughput=tput))
    print(f"{n:>10,} {mn:>10.3f} {mx:>10.3f} {mean:>10.3f} {std:>10.4f} "
          f"{tput:>10.2f} {improvement:>9.2f}x")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2 — Kernel-Only Latency
# Data pre-transferred to the device before timing begins.
# Isolates pure GPU compute time from PCIe transfer overhead.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 2 — Kernel-Only Latency (device-resident data)")
print(f"  {REPEATS} repeats × {NUMBER} calls each\n")
print(f"{'Size':>10} {'Min(ms)':>10} {'Max(ms)':>10} {'Mean(ms)':>10} "
      f"{'Std(ms)':>10} {'Tput M/s':>10}")
print("─" * 62)

kernel_results = []
for n in SIZES:
    inp = make_inputs(n)
    d   = make_device_inputs(inp)
    bpg = d["bpg"]

    def run_kernel():
        black_scholes_kernel[bpg, THREADSPERBLOCK](
            d["d_S"], d["d_X"], d["d_T"], d["d_R"], d["d_V"],
            d["d_call"], d["d_put"])
        cuda.synchronize()

    run_kernel()

    raw = timeit.repeat(
        stmt    = "run_kernel()",
        setup   = "pass",
        globals = {"run_kernel": run_kernel},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    per_call_ms = [t / NUMBER * 1000 for t in raw]
    mn = min(per_call_ms); mx = max(per_call_ms)
    mean = statistics.mean(per_call_ms); std = statistics.stdev(per_call_ms)
    tput = n / (mean / 1000) / 1e6
    kernel_results.append(dict(size=n, min=mn, max=mx, mean=mean, std=std, throughput=tput))
    print(f"{n:>10,} {mn:>10.3f} {mx:>10.3f} {mean:>10.3f} {std:>10.4f} {tput:>10.2f}")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3 — PCIe Transfer Breakdown (H2D = Host-to-Device) (D2H = Device-to-Host)
# H2D and D2H transfers measured independently to quantify what fraction
# of end-to-end latency is attributable to data movement.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 3 — PCIe Transfer Breakdown (H2D and D2H)\n")
print(f"{'Size':>10} {'H2D(ms)':>12} {'D2H(ms)':>12} {'Transfer%(e2e)':>16}")
print("─" * 54)

for idx, n in enumerate(SIZES):
    inp = make_inputs(n)
    _ = cuda.to_device(inp["stockPrice"]); cuda.synchronize()

    raw_h2d = timeit.repeat(
        stmt = (
            "cuda.to_device(inp['stockPrice']);"
            "cuda.to_device(inp['optionStrike']);"
            "cuda.to_device(inp['optionYears']);"
            "cuda.to_device(inp['Riskfree']);"
            "cuda.to_device(inp['Volatility']);"
            "cuda.synchronize()"
        ),
        setup   = "pass",
        globals = {"cuda": cuda, "inp": inp},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    h2d_ms = min(raw_h2d) / NUMBER * 1000

    d_call = cuda.device_array(n, dtype=np.float64)
    d_put  = cuda.device_array(n, dtype=np.float64)
    d_call.copy_to_host(); cuda.synchronize()

    raw_d2h = timeit.repeat(
        stmt    = "d_call.copy_to_host(); d_put.copy_to_host(); cuda.synchronize()",
        setup   = "pass",
        globals = {"cuda": cuda, "d_call": d_call, "d_put": d_put},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    d2h_ms = min(raw_d2h) / NUMBER * 1000

    transfer_pct = (h2d_ms + d2h_ms) / e2e_results[idx]["min"] * 100
    print(f"{n:>10,} {h2d_ms:>12.3f} {d2h_ms:>12.3f} {transfer_pct:>16.1f}%")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4 — GPU vs CPU Speedup Summary
# Compares pageable, pinned, and kernel-only GPU paths against the
# vectorised NumPy CPU baseline for each array size.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 4 — GPU vs NumPy CPU Speedup\n")
print(f"{'Size':>10} {'CPU(ms)':>10} {'E2E-Page':>10} {'E2E-Pin':>9} "
      f"{'Kern(ms)':>10} {'Spd-Page':>10} {'Spd-Pin':>9} {'Spd-Kern':>10}")
print("─" * 84)

for idx, n in enumerate(SIZES):
    inp = make_inputs(n)
    black_scholes_cpu(**inp)

    cpu_raw = timeit.repeat(
        stmt    = "black_scholes_cpu(**inp)",
        setup   = "pass",
        globals = {"black_scholes_cpu": black_scholes_cpu, "inp": inp},
        repeat  = REPEATS,
        number  = NUMBER,
    )
    cpu_ms = min(t / NUMBER * 1000 for t in cpu_raw)

    gpu_e2e_page = e2e_results[idx]["min"]
    gpu_e2e_pin  = e2e_pinned_results[idx]["min"]
    gpu_kern_ms  = kernel_results[idx]["min"]

    print(f"{n:>10,} {cpu_ms:>10.3f} {gpu_e2e_page:>10.3f} {gpu_e2e_pin:>9.3f} "
          f"{gpu_kern_ms:>10.3f} {cpu_ms/gpu_e2e_page:>9.2f}x "
          f"{cpu_ms/gpu_e2e_pin:>8.2f}x {cpu_ms/gpu_kern_ms:>9.2f}x")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5 — Numerical Correctness Verification
# Confirms GPU and CPU results agree within floating-point tolerances.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 5 — Numerical Correctness Verification\n")
inp = make_inputs(1_000_000)
call_gpu, put_gpu = black_scholes_optimized(**inp)
call_cpu, put_cpu = black_scholes_cpu(**inp)
np.testing.assert_allclose(call_gpu, call_cpu, rtol=1e-5, atol=1e-8)
np.testing.assert_allclose(put_gpu,  put_cpu,  rtol=1e-5, atol=1e-8)
print("✓ GPU and CPU results agree within tolerances (rtol=1e-5, atol=1e-8)")
print(f"  Max absolute error — Call: {np.max(np.abs(call_gpu - call_cpu)):.2e}"
      f"  |  Put: {np.max(np.abs(put_gpu - put_cpu)):.2e}")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 6 — GPU-Resident vs CPU-Resident Batch Pricing
# Both sides pre-allocate output buffers once and reuse them across the
# entire batch. The GPU additionally amortises the one-time PCIe transfer
# cost over BATCH_SIZE kernel runs.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"PHASE 6 — Resident Batch Pricing  (batch size = {BATCH_SIZE})")
print("  GPU: transfer once → kernel × N → copy back once")
print("  CPU: pre-allocate outputs once → compute × N  (no alloc per call)\n")
print(f"{'Size':>10} {'CPU-Res(ms)':>13} {'GPU-Res(ms)':>13} "
      f"{'Eff.Spd':>9} {'vs Phase1':>10}")
print("─" * 62)

for idx, n in enumerate(SIZES):

    # ── CPU-resident setup: pre-allocate inputs and output buffers ────
    inp_cpu  = make_inputs(n)
    S, X     = inp_cpu["stockPrice"],   inp_cpu["optionStrike"]
    T, R, V  = inp_cpu["optionYears"],  inp_cpu["Riskfree"], inp_cpu["Volatility"]
    call_out = np.empty(n, dtype=np.float64)   # pre-allocated output — reused every call
    put_out  = np.empty(n, dtype=np.float64)   # pre-allocated output — reused every call

    # CPU warm-up
    black_scholes_cpu_resident(S, X, T, R, V, call_out, put_out)

    t0 = time.perf_counter()
    for _ in range(BATCH_SIZE):
        black_scholes_cpu_resident(S, X, T, R, V, call_out, put_out)
    t_cpu_res = (time.perf_counter() - t0) * 1000

    # ── GPU-resident setup: pinned transfer once, pre-allocate outputs ─
    inp_p  = make_inputs_pinned(n)
    d      = make_device_inputs_pinned(inp_p)
    bpg    = d["bpg"]

    # GPU warm-up
    black_scholes_gpu_resident(
        d["d_S"], d["d_X"], d["d_T"], d["d_R"], d["d_V"],
        d["d_call"], d["d_put"], bpg)
    cuda.synchronize()

    t0     = time.perf_counter()
    d_S    = cuda.to_device(inp_p["stockPrice"])
    d_X    = cuda.to_device(inp_p["optionStrike"])
    d_T    = cuda.to_device(inp_p["optionYears"])
    d_R    = cuda.to_device(inp_p["Riskfree"])
    d_V    = cuda.to_device(inp_p["Volatility"])
    d_call = cuda.device_array(n, dtype=np.float64)
    d_put  = cuda.device_array(n, dtype=np.float64)

    for _ in range(BATCH_SIZE):
        black_scholes_gpu_resident(d_S, d_X, d_T, d_R, d_V, d_call, d_put, bpg)
    cuda.synchronize()

    d_call.copy_to_host()
    d_put.copy_to_host()
    t_gpu_res = (time.perf_counter() - t0) * 1000

    eff_speedup = t_cpu_res / t_gpu_res
    vs_phase1   = e2e_results[idx]["mean"] * BATCH_SIZE / t_gpu_res

    print(f"{n:>10,} {t_cpu_res:>13.1f} {t_gpu_res:>13.1f} "
          f"{eff_speedup:>8.1f}x {vs_phase1:>9.1f}x")

print(f"\n  Effective speedup converges toward the kernel-only limit as batch size grows.")
print(f"  'vs Phase1' column shows improvement over {BATCH_SIZE} × individual calls.")


GPU                : Tesla T4
Compute Capability : (7, 5)
Max Threads/Block  : 1024
Multiprocessors    : 40

Compiling CUDA kernel ...
Compilation complete.

PHASE 1 — End-to-End Latency: Pageable Memory
  10 repeats × 5 calls each

      Size    Min(ms)    Max(ms)   Mean(ms)    Std(ms)   Tput M/s
──────────────────────────────────────────────────────────────
       100      1.307      1.613      1.372     0.0871       0.07
     1,000      1.345      1.753      1.489     0.1341       0.67
    10,000      1.502      1.780      1.584     0.0753       6.31
   100,000      4.066      4.317      4.162     0.0801      24.03
 1,000,000     18.338     21.324     19.974     1.0550      50.07
10,000,000    169.691    175.773    172.790     1.8587      57.87

PHASE 1b — End-to-End Latency: Pinned Memory
  10 repeats × 5 calls each

      Size    Min(ms)    Max(ms)   Mean(ms)    Std(ms)   Tput M/s      vs 1a
──────────────────────────────────────────────────────────────────────────
       100     

Here is a complete breakdown of every phase in above results.

***
## GPU Hardware: Tesla T4
The T4 has 40 SMs with Compute Capability 7.5 (Turing), supporting up to 1024 threads/block . With `THREADSPERBLOCK=256`, you need at least 80 active blocks (`2 × 40 SMs`) to fully saturate the GPU - that requires `n ≥ 20,480` elements, which explains the poor throughput at n=100 and n=1,000.

***
## Phase 1 vs Phase 1b - Pinned Memory Impact
Pinned memory delivers **no meaningful gain below n=10,000** (0.97-1.01×) because the fixed CUDA driver overhead (~1.3 ms) completely overwhelms the transfer itself. The gains only emerge where data volume becomes large enough for the DMA bandwidth difference to matter - **1.25× at n=100k, 1.58× at n=1M, 1.42× at n=10M**. The slight dip to 1.42× at 10M (vs 1.58× at 1M) is because at very large sizes, PCIe bandwidth saturation starts affecting both pageable and pinned transfers, narrowing the gap.

***
## Phase 2 - Kernel Ceiling
The kernel-only throughput plateaus at **692 M options/sec at n=10M** (vs 668 M/s at n=1M), confirming the GPU is fully saturated and memory-bandwidth-bound inside the device . There is nothing left to squeeze from the kernel itself - the 87× speedup at n=10M is the hard ceiling this GPU can deliver.

***
## Phase 3 - PCIe Is the Real Bottleneck
Transfer accounts for **75-90% of every single end-to-end call**. One important detail: D2H at n=10M (59ms) is nearly as expensive as H2D (93ms) despite reading only 2 arrays versus writing 5. This is because the T4 in a cloud VM uses pageable host memory for the output, requiring an extra CPU memcpy from a pinned staging buffer before results reach your NumPy array.

***
## Phase 4 - Full Speedup Summary
Three distinct performance regimes are visible:

| Size range | Winner | Why |
|---|---|---|
| n ≤ 10,000 | **CPU** | PCIe fixed overhead (~1.4 ms) exceeds total CPU compute time |
| n = 100,000 | **GPU wins** | Pinned: 1.85×, Pageable: 1.48× - compute starts paying off |
| n ≥ 1,000,000 | **GPU dominant** | Pinned reaches 10.41×, kernel-only 87× |

The break-even point shifted from ~100k (pageable) to somewhere between 10k and 100k with pinned memory - pinned memory moved the crossover point to a smaller problem size.

***
## Phase 6 - The Most Important Result
This is where everything comes together. At **n=10M with batch=100**, the GPU-resident approach processes 1 billion options in 1,653 ms while CPU-resident takes 129,130 ms - a **78.1× real-world speedup**, compared to only 7.32× in the naive single-call approach. The "vs Phase1" column tells you how much you left on the table with the naive approach: at n=10M, batch pricing is **10.5× faster than making 100 individual calls**.

The progression toward the 87× kernel ceiling as array size grows is clear:

| Size | Batch Speedup | % of kernel ceiling (87×) |
|------|:------------:|:-------------------------:|
| 100 | 0.9× | 1% |
| 1,000 | 1.5× | 2% |
| 10,000 | 6.2× | 7% |
| 100,000 | 15.5× | 18% |
| 1,000,000 | 32.1× | 37% |
| 10,000,000 | **78.1×** | **90%** |

The batch-resident strategy at n=10M recovers **90% of the theoretical kernel-ceiling speedup** in a real end-to-end scenario - the remaining 10% gap is the amortised PCIe cost of the one-time transfer divided across 100 runs.

# Code Sample 2

## Pinned Memory

In [6]:
import numpy as np
import time
from numba import cuda
import math

# --- CPU Version ---
def cnd_cpu(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S = stockPrice
    X = optionStrike
    T = optionYears
    R = Riskfree
    V = Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1)
    cndd2 = cnd_cpu(d2)

    expRT = np.exp(- R * T)

    callResult = S * cndd1 - X * expRT * cndd2
    putResult = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

    return callResult, putResult

# --- GPU Version ---
@cuda.jit(device=True, inline=True, fastmath=True)
def cnd_device(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d))
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    if d > 0:
        return 1.0 - ret_val
    else:
        return ret_val

@cuda.jit(fastmath=True)
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility, callResult, putResult):
    tid = cuda.grid(1)
    N = stockPrice.shape[0]

    if tid < N:
        S = stockPrice[tid]
        X = optionStrike[tid]
        T = optionYears[tid]
        R = Riskfree[tid]
        V = Volatility[tid]

        sqrtT = math.sqrt(T)
        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
        d2 = d1 - V * sqrtT

        cndd1 = cnd_device(d1)
        cndd2 = cnd_device(d2)

        expRT = math.exp(- R * T)

        callResult[tid] = S * cndd1 - X * expRT * cndd2
        putResult[tid] = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# --- Benchmarking Setup ---
def run_benchmark(num_options=10**7, iterations=5):
    # Generate random input data
    np.random.seed(42) # for reproducibility
    stockPrice = np.random.uniform(50, 150, num_options).astype(np.float64)
    optionStrike = np.random.uniform(50, 150, num_options).astype(np.float64)
    optionYears = np.random.uniform(0.5, 2.0, num_options).astype(np.float64)
    Riskfree = np.random.uniform(0.01, 0.05, num_options).astype(np.float64)
    Volatility = np.random.uniform(0.1, 0.5, num_options).astype(np.float64)

    print(f"Benchmarking with {num_options} options over {iterations} iterations.")

    # --- CPU Benchmark ---
    cpu_times = []
    cpu_call_result = None
    cpu_put_result = None

    print("\n--- CPU Warm Up Started ---")
    # Warm Up CPU
    cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility
        )
    print("\n--- CPU Warm Up Done ---")
    
    print("\n--- Starting CPU Benchmark ---")
    for i in range(iterations):
        start_time = time.perf_counter()
        cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility
        )
        end_time = time.perf_counter()
        cpu_times.append(end_time - start_time)
        print(f"CPU iteration {i+1}: {cpu_times[-1]:.4f} seconds")
    avg_cpu_time = np.mean(cpu_times)
    print(f"Average CPU time: {avg_cpu_time:.4f} seconds")

    # --- GPU Benchmark ---
    if not cuda.is_available():
        print("\nCUDA is not available. Skipping GPU benchmark.")
        return

    gpu_times = []

    # # ── WINDOW 1: Host → Device (H2D) transfer only ──────────────────────────
    # t0 = time.perf_counter()
    
    # # Allocate device memory
    # d_stockPrice = cuda.to_device(stockPrice)
    # d_optionStrike = cuda.to_device(optionStrike)
    # d_optionYears = cuda.to_device(optionYears)
    # d_Riskfree = cuda.to_device(Riskfree)
    # d_Volatility = cuda.to_device(Volatility)
    # d_callResult = cuda.device_array_like(stockPrice)
    # d_putResult = cuda.device_array_like(stockPrice)

    # cuda.synchronize()               # ← MUST sync; to_device is async
    # t1 = time.perf_counter()
    # h2d_time = t1 - t0
    # print(f"H2D Transfer:    {h2d_time:.4f}s")

    # gpu_call_result = np.empty_like(stockPrice)
    # gpu_put_result = np.empty_like(stockPrice)

    # Step 1: Allocate pinned buffers for inputs (outside H2D timing window)
    pin_stockPrice   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionStrike = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionYears  = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Riskfree     = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Volatility   = cuda.pinned_array(num_options, dtype=np.float64)
    
    # Step 2: Copy pageable → pinned (outside H2D timing window)
    pin_stockPrice[:]   = stockPrice       # pageable → pinned (CPU memcpy)
    pin_optionStrike[:] = optionStrike
    pin_optionYears[:]  = optionYears
    pin_Riskfree[:]     = Riskfree
    pin_Volatility[:]   = Volatility
    
    # Step 3: Allocate pinned OUTPUT buffers for D2H (outside both windows)
    gpu_call_result = cuda.pinned_array(num_options, dtype=np.float64)
    gpu_put_result  = cuda.pinned_array(num_options, dtype=np.float64)
    
    # Step 4: Allocate GPU-side result arrays (outside H2D timing window)
    d_callResult = cuda.device_array(num_options, dtype=np.float64)
    d_putResult  = cuda.device_array(num_options, dtype=np.float64)
    
    # ── WINDOW 1: H2D — now uses pinned memory, much faster ──────────────────
    t0 = time.perf_counter()
    d_stockPrice   = cuda.to_device(pin_stockPrice)    # pinned → GPU (fast DMA)
    d_optionStrike = cuda.to_device(pin_optionStrike)
    d_optionYears  = cuda.to_device(pin_optionYears)
    d_Riskfree     = cuda.to_device(pin_Riskfree)
    d_Volatility   = cuda.to_device(pin_Volatility)
    cuda.synchronize()
    t1 = time.perf_counter()
    print(f"H2D Transfer: {t1 - t0:.4f}s")

    

    # Determine kernel launch configuration
    threadsperblock = 256 # A common choice for good occupancy, adjust based on GPU architecture
    blockspergrid = (num_options + threadsperblock - 1) // threadsperblock
    print(f"GPU Launch Config: {blockspergrid} blocks, {threadsperblock} threads per block")

    
    print("\n--- GPU Warm Up Started ---")
    for i in range(3):
        black_scholes_kernel[blockspergrid, threadsperblock](
                d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
                d_callResult, d_putResult
            )
        cuda.synchronize() # Wait for GPU to finish
    print("\n--- GPU Warm Up Done ---")
    print("\n--- Starting GPU Benchmark ---")

    for i in range(iterations):
        # ── WINDOW 2: Pure kernel time (what you already have) ───────────────────
        start_time = time.perf_counter()
        black_scholes_kernel[blockspergrid, threadsperblock](
            d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
            d_callResult, d_putResult
        )
        cuda.synchronize() # Wait for GPU to finish
        end_time = time.perf_counter()
        gpu_times.append(end_time - start_time)
        print(f"GPU iteration {i+1}: {gpu_times[-1]:.4f} seconds")
        
    avg_gpu_time = np.mean(gpu_times)
    print(f"Average GPU time: {avg_gpu_time:.4f} seconds")



    
    # ── WINDOW 3: Device → Host (D2H) transfer only ──────────────────────────
    t2 = time.perf_counter()
    # Transfer results back to host for verification
    d_callResult.copy_to_host(gpu_call_result)
    d_putResult.copy_to_host(gpu_put_result)
    cuda.synchronize()
    t3 = time.perf_counter()
    d2h_time = t3 - t2
    print(f"D2H Transfer:    {d2h_time:.4f}s")

    
    # --- Verification ---
    if cpu_call_result is not None and cpu_put_result is not None:
        print("\n--- Verifying Results ---")
        # Check a small subset for numerical stability differences
        sample_indices = np.random.choice(num_options, min(10, num_options), replace=False)

        print("\nSample Call Results:")
        print("CPU:", cpu_call_result[sample_indices])
        print("GPU:", gpu_call_result[sample_indices])

        print("\nSample Put Results:")
        print("CPU:", cpu_put_result[sample_indices])
        print("GPU:", gpu_put_result[sample_indices])

        # Use allclose for floating point comparison
        # Increased rtol/atol slightly due to fastmath and different math library implementations
        call_diff = np.allclose(cpu_call_result, gpu_call_result, rtol=1e-4, atol=1e-6)
        put_diff = np.allclose(cpu_put_result, gpu_put_result, rtol=1e-4, atol=1e-6)

        if call_diff and put_diff:
            print("Results match between CPU and GPU (within tolerance).")
        else:
            print("WARNING: Results do NOT perfectly match between CPU and GPU.")
            if not call_diff:
                print("Call results differ.")
            if not put_diff:
                print("Put results differ.")
            # Print max absolute difference for debugging
            print(f"Max absolute difference in Call results: {np.max(np.abs(cpu_call_result - gpu_call_result)):.2e}")
            print(f"Max absolute difference in Put results: {np.max(np.abs(cpu_put_result - gpu_put_result)):.2e}")

    if cuda.is_available():
        if avg_gpu_time > 0: # Avoid division by zero if GPU time is somehow 0
            speedup = avg_cpu_time / avg_gpu_time
            print(f"\n--- Performance Summary ---")
            print(f"GPU is {speedup:.2f}x faster than CPU.")
        else:
            print("\nGPU time was zero, cannot calculate speedup.")

# Run the benchmark 
run_benchmark(num_options=(10**8) * 1, iterations=5)

Benchmarking with 100000000 options over 5 iterations.

--- CPU Warm Up Started ---

--- CPU Warm Up Done ---

--- Starting CPU Benchmark ---
CPU iteration 1: 13.1226 seconds
CPU iteration 2: 13.1409 seconds
CPU iteration 3: 13.1523 seconds
CPU iteration 4: 13.1594 seconds
CPU iteration 5: 13.1405 seconds
Average CPU time: 13.1431 seconds
H2D Transfer: 0.3282s
GPU Launch Config: 390625 blocks, 256 threads per block

--- GPU Warm Up Started ---

--- GPU Warm Up Done ---

--- Starting GPU Benchmark ---
GPU iteration 1: 0.1340 seconds
GPU iteration 2: 0.1340 seconds
GPU iteration 3: 0.1340 seconds
GPU iteration 4: 0.1341 seconds
GPU iteration 5: 0.1341 seconds
Average GPU time: 0.1341 seconds
D2H Transfer:    0.1224s

--- Verifying Results ---

Sample Call Results:
CPU: [10.97212405  8.98799198 49.10857824 21.64761811 13.31766179 12.60110805
 25.82739995 29.85409807  2.31476154 43.02943625]
GPU: [10.97212405  8.98799198 49.10857824 21.64761811 13.31766179 12.60110805
 25.82739995 29.85409

In [1]:
import numpy as np
import time
from numba import cuda
import math

# --- CPU Version ---
def cnd_cpu(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S = stockPrice
    X = optionStrike
    T = optionYears
    R = Riskfree
    V = Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1)
    cndd2 = cnd_cpu(d2)

    expRT = np.exp(- R * T)

    callResult = S * cndd1 - X * expRT * cndd2
    putResult = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

    return callResult, putResult

# --- GPU Version ---
@cuda.jit(device=True, inline=True, fastmath=True)
def cnd_device(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d))
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    if d > 0:
        return 1.0 - ret_val
    else:
        return ret_val

@cuda.jit(fastmath=True)
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility, callResult, putResult):
    tid = cuda.grid(1)
    N = stockPrice.shape[0]

    if tid < N:
        S = stockPrice[tid]
        X = optionStrike[tid]
        T = optionYears[tid]
        R = Riskfree[tid]
        V = Volatility[tid]

        sqrtT = math.sqrt(T)
        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
        d2 = d1 - V * sqrtT

        cndd1 = cnd_device(d1)
        cndd2 = cnd_device(d2)

        expRT = math.exp(- R * T)

        callResult[tid] = S * cndd1 - X * expRT * cndd2
        putResult[tid] = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# --- Benchmarking Setup ---
def run_benchmark(num_options=10**7, iterations=5):
    # Generate random input data
    np.random.seed(42) # for reproducibility
    stockPrice = np.random.uniform(50, 150, num_options).astype(np.float64)
    optionStrike = np.random.uniform(50, 150, num_options).astype(np.float64)
    optionYears = np.random.uniform(0.5, 2.0, num_options).astype(np.float64)
    Riskfree = np.random.uniform(0.01, 0.05, num_options).astype(np.float64)
    Volatility = np.random.uniform(0.1, 0.5, num_options).astype(np.float64)

    print(f"Benchmarking with {num_options} options over {iterations} iterations.")

    # --- CPU Benchmark ---
    cpu_times = []
    cpu_call_result = None
    cpu_put_result = None

    print("\n--- CPU Warm Up Started ---")
    # Warm Up CPU
    cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility
        )
    print("\n--- CPU Warm Up Done ---")
    
    print("\n--- Starting CPU Benchmark ---")
    for i in range(iterations):
        start_time = time.perf_counter()
        cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility
        )
        end_time = time.perf_counter()
        cpu_times.append(end_time - start_time)
        print(f"CPU iteration {i+1}: {cpu_times[-1]:.4f} seconds")
    avg_cpu_time = np.mean(cpu_times)
    print(f"Average CPU time: {avg_cpu_time:.4f} seconds")

    # --- GPU Benchmark ---
    if not cuda.is_available():
        print("\nCUDA is not available. Skipping GPU benchmark.")
        return

    gpu_times = []

    # # ── WINDOW 1: Host → Device (H2D) transfer only ──────────────────────────
    # t0 = time.perf_counter()
    
    # # Allocate device memory
    # d_stockPrice = cuda.to_device(stockPrice)
    # d_optionStrike = cuda.to_device(optionStrike)
    # d_optionYears = cuda.to_device(optionYears)
    # d_Riskfree = cuda.to_device(Riskfree)
    # d_Volatility = cuda.to_device(Volatility)
    # d_callResult = cuda.device_array_like(stockPrice)
    # d_putResult = cuda.device_array_like(stockPrice)

    # cuda.synchronize()               # ← MUST sync; to_device is async
    # t1 = time.perf_counter()
    # h2d_time = t1 - t0
    # print(f"H2D Transfer:    {h2d_time:.4f}s")

    # gpu_call_result = np.empty_like(stockPrice)
    # gpu_put_result = np.empty_like(stockPrice)

    # Step 1: Allocate pinned buffers for inputs (outside H2D timing window)
    pin_stockPrice   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionStrike = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionYears  = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Riskfree     = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Volatility   = cuda.pinned_array(num_options, dtype=np.float64)
    
    # Step 2: Copy pageable → pinned (outside H2D timing window)
    pin_stockPrice[:]   = stockPrice       # pageable → pinned (CPU memcpy)
    pin_optionStrike[:] = optionStrike
    pin_optionYears[:]  = optionYears
    pin_Riskfree[:]     = Riskfree
    pin_Volatility[:]   = Volatility
    
    # Step 3: Allocate pinned OUTPUT buffers for D2H (outside both windows)
    gpu_call_result = cuda.pinned_array(num_options, dtype=np.float64)
    gpu_put_result  = cuda.pinned_array(num_options, dtype=np.float64)
    
    # Step 4: Allocate GPU-side result arrays (outside H2D timing window)
    d_callResult = cuda.device_array(num_options, dtype=np.float64)
    d_putResult  = cuda.device_array(num_options, dtype=np.float64)
    
    # ── WINDOW 1: H2D — now uses pinned memory, much faster ──────────────────
    t0 = time.perf_counter()
    d_stockPrice   = cuda.to_device(pin_stockPrice)    # pinned → GPU (fast DMA)
    d_optionStrike = cuda.to_device(pin_optionStrike)
    d_optionYears  = cuda.to_device(pin_optionYears)
    d_Riskfree     = cuda.to_device(pin_Riskfree)
    d_Volatility   = cuda.to_device(pin_Volatility)
    cuda.synchronize()
    t1 = time.perf_counter()
    print(f"H2D Transfer: {t1 - t0:.4f}s")

    

    # Determine kernel launch configuration
    threadsperblock = 256 # A common choice for good occupancy, adjust based on GPU architecture
    blockspergrid = (num_options + threadsperblock - 1) // threadsperblock
    print(f"GPU Launch Config: {blockspergrid} blocks, {threadsperblock} threads per block")

    
    print("\n--- GPU Warm Up Started ---")
    for i in range(3):
        black_scholes_kernel[blockspergrid, threadsperblock](
                d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
                d_callResult, d_putResult
            )
        cuda.synchronize() # Wait for GPU to finish
    print("\n--- GPU Warm Up Done ---")
    print("\n--- Starting GPU Benchmark ---")

    for i in range(iterations):
        # ── WINDOW 2: Pure kernel time (what you already have) ───────────────────
        start_time = time.perf_counter()
        black_scholes_kernel[blockspergrid, threadsperblock](
            d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
            d_callResult, d_putResult
        )
        cuda.synchronize() # Wait for GPU to finish
        end_time = time.perf_counter()
        gpu_times.append(end_time - start_time)
        print(f"GPU iteration {i+1}: {gpu_times[-1]:.4f} seconds")
        
    avg_gpu_time = np.mean(gpu_times)
    print(f"Average GPU time: {avg_gpu_time:.4f} seconds")



    
    # ── WINDOW 3: Device → Host (D2H) transfer only ──────────────────────────
    t2 = time.perf_counter()
    # Transfer results back to host for verification
    d_callResult.copy_to_host(gpu_call_result)
    d_putResult.copy_to_host(gpu_put_result)
    cuda.synchronize()
    t3 = time.perf_counter()
    d2h_time = t3 - t2
    print(f"D2H Transfer:    {d2h_time:.4f}s")

    
    # --- Verification ---
    if cpu_call_result is not None and cpu_put_result is not None:
        print("\n--- Verifying Results ---")
        # Check a small subset for numerical stability differences
        sample_indices = np.random.choice(num_options, min(10, num_options), replace=False)

        print("\nSample Call Results:")
        print("CPU:", cpu_call_result[sample_indices])
        print("GPU:", gpu_call_result[sample_indices])

        print("\nSample Put Results:")
        print("CPU:", cpu_put_result[sample_indices])
        print("GPU:", gpu_put_result[sample_indices])

        # Use allclose for floating point comparison
        # Increased rtol/atol slightly due to fastmath and different math library implementations
        call_diff = np.allclose(cpu_call_result, gpu_call_result, rtol=1e-4, atol=1e-6)
        put_diff = np.allclose(cpu_put_result, gpu_put_result, rtol=1e-4, atol=1e-6)

        if call_diff and put_diff:
            print("Results match between CPU and GPU (within tolerance).")
        else:
            print("WARNING: Results do NOT perfectly match between CPU and GPU.")
            if not call_diff:
                print("Call results differ.")
            if not put_diff:
                print("Put results differ.")
            # Print max absolute difference for debugging
            print(f"Max absolute difference in Call results: {np.max(np.abs(cpu_call_result - gpu_call_result)):.2e}")
            print(f"Max absolute difference in Put results: {np.max(np.abs(cpu_put_result - gpu_put_result)):.2e}")

    if cuda.is_available():
        if avg_gpu_time > 0: # Avoid division by zero if GPU time is somehow 0
            speedup = avg_cpu_time / avg_gpu_time
            print(f"\n--- Performance Summary ---")
            print(f"GPU is {speedup:.2f}x faster than CPU.")
        else:
            print("\nGPU time was zero, cannot calculate speedup.")

# Run the benchmark 
run_benchmark(num_options=(10**8) * 2, iterations=5)

Benchmarking with 200000000 options over 5 iterations.

--- CPU Warm Up Started ---

--- CPU Warm Up Done ---

--- Starting CPU Benchmark ---
CPU iteration 1: 26.1074 seconds
CPU iteration 2: 26.1660 seconds
CPU iteration 3: 26.0708 seconds
CPU iteration 4: 26.1973 seconds
CPU iteration 5: 26.1161 seconds
Average CPU time: 26.1315 seconds
H2D Transfer: 0.6530s
GPU Launch Config: 781250 blocks, 256 threads per block

--- GPU Warm Up Started ---

--- GPU Warm Up Done ---

--- Starting GPU Benchmark ---
GPU iteration 1: 0.2677 seconds
GPU iteration 2: 0.2677 seconds
GPU iteration 3: 0.2679 seconds
GPU iteration 4: 0.2677 seconds
GPU iteration 5: 0.2677 seconds
Average GPU time: 0.2677 seconds
D2H Transfer:    0.2441s

--- Verifying Results ---

Sample Call Results:
CPU: [10.0332485  84.86172617  0.8517943  40.12633017  1.49828782 41.38112611
 46.07402779 32.82489325 61.25152044 18.40847933]
GPU: [10.0332485  84.86172617  0.8517943  40.12633017  1.49828782 41.38112611
 46.07402779 32.82489

## CUDA STREAMS + Pinned Memory

In [1]:
import numpy as np
import time
from numba import cuda
import math

# --- CPU Version (unchanged) ---
def cnd_cpu(d):
    A1 = 0.31938153; A2 = -0.356563782; A3 = 1.781477937
    A4 = -1.821255978; A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S, X, T, R, V = stockPrice, optionStrike, optionYears, Riskfree, Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1); cndd2 = cnd_cpu(d2)
    expRT = np.exp(-R * T)
    return S * cndd1 - X * expRT * cndd2, X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# --- GPU Kernel (unchanged) ---
@cuda.jit(device=True, inline=True, fastmath=True)
def cnd_device(d):
    A1 = 0.31938153; A2 = -0.356563782; A3 = 1.781477937
    A4 = -1.821255978; A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d))
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return (1.0 - ret_val) if d > 0 else ret_val

@cuda.jit(fastmath=True)
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility,
                          callResult, putResult):
    tid = cuda.grid(1)
    N = stockPrice.shape[0]
    if tid < N:
        S, X, T, R, V = stockPrice[tid], optionStrike[tid], optionYears[tid], Riskfree[tid], Volatility[tid]
        sqrtT = math.sqrt(T)
        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
        d2 = d1 - V * sqrtT
        cndd1 = cnd_device(d1); cndd2 = cnd_device(d2)
        expRT = math.exp(-R * T)
        callResult[tid] = S * cndd1 - X * expRT * cndd2
        putResult[tid]  = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)


def run_benchmark(num_options=10**7, iterations=5, num_streams=4):

    np.random.seed(42)
    stockPrice   = np.random.uniform(50,   150,  num_options).astype(np.float64)
    optionStrike = np.random.uniform(50,   150,  num_options).astype(np.float64)
    optionYears  = np.random.uniform(0.5,  2.0,  num_options).astype(np.float64)
    Riskfree     = np.random.uniform(0.01, 0.05, num_options).astype(np.float64)
    Volatility   = np.random.uniform(0.1,  0.5,  num_options).astype(np.float64)

    print(f"Benchmarking with {num_options} options, {iterations} iterations, {num_streams} streams.")

    # ── CPU Benchmark ─────────────────────────────────────────────────────────
    cpu_call_result, cpu_put_result = black_scholes_cpu(
        stockPrice, optionStrike, optionYears, Riskfree, Volatility)  # warm-up

    cpu_times = []
    print("\n--- Starting CPU Benchmark ---")
    for i in range(iterations):
        t0 = time.perf_counter()
        cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility)
        cpu_times.append(time.perf_counter() - t0)
        print(f"  CPU iteration {i+1}: {cpu_times[-1]:.4f}s")
    avg_cpu_time = np.mean(cpu_times)
    print(f"Average CPU time: {avg_cpu_time:.4f}s")

    if not cuda.is_available():
        print("\nCUDA not available. Skipping GPU benchmark.")
        return

    # ── Pinned Memory Allocation ──────────────────────────────────────────────
    # Input arrays: pinned pageable→pinned copy
    pin_stockPrice   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionStrike = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionYears  = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Riskfree     = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Volatility   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_stockPrice[:]   = stockPrice
    pin_optionStrike[:] = optionStrike
    pin_optionYears[:]  = optionYears
    pin_Riskfree[:]     = Riskfree
    pin_Volatility[:]   = Volatility

    # Output arrays: pinned so D2H is also fast
    gpu_call_result = cuda.pinned_array(num_options, dtype=np.float64)
    gpu_put_result  = cuda.pinned_array(num_options, dtype=np.float64)

    # ── Stream Setup ──────────────────────────────────────────────────────────
    # Divide work evenly; last chunk absorbs any remainder
    chunk_size = (num_options + num_streams - 1) // num_streams

    streams = [cuda.stream() for _ in range(num_streams)]

    # Pre-allocate one set of GPU arrays per stream (reused every iteration)
    # This avoids repeated GPU malloc inside the timed loop
    d_in  = []   # list of [d_S, d_X, d_T, d_R, d_V] per stream
    d_out = []   # list of [d_call, d_put]           per stream

    for s in range(num_streams):
        start = s * chunk_size
        end   = min(start + chunk_size, num_options)
        size  = end - start
        d_in.append([
            cuda.device_array(size, dtype=np.float64),  # stockPrice
            cuda.device_array(size, dtype=np.float64),  # optionStrike
            cuda.device_array(size, dtype=np.float64),  # optionYears
            cuda.device_array(size, dtype=np.float64),  # Riskfree
            cuda.device_array(size, dtype=np.float64),  # Volatility
        ])
        d_out.append([
            cuda.device_array(size, dtype=np.float64),  # callResult
            cuda.device_array(size, dtype=np.float64),  # putResult
        ])

    tpb = 256  # threads per block

    # Helper: dispatch one full pipeline pass across all streams
    def dispatch_all_streams():
        for s in range(num_streams):
            start  = s * chunk_size
            end    = min(start + chunk_size, num_options)
            size   = end - start
            blocks = (size + tpb - 1) // tpb
            st     = streams[s]
            si, so = d_in[s], d_out[s]

            # ① Async H2D: pinned host chunk → GPU (no CPU stall)
            si[0].copy_to_device(pin_stockPrice[start:end],   stream=st)
            si[1].copy_to_device(pin_optionStrike[start:end], stream=st)
            si[2].copy_to_device(pin_optionYears[start:end],  stream=st)
            si[3].copy_to_device(pin_Riskfree[start:end],     stream=st)
            si[4].copy_to_device(pin_Volatility[start:end],   stream=st)

            # ② Async Kernel: runs after H2D completes ON THIS STREAM
            black_scholes_kernel[blocks, tpb, st](
                si[0], si[1], si[2], si[3], si[4], so[0], so[1])

            # ③ Async D2H: runs after kernel completes ON THIS STREAM
            so[0].copy_to_host(gpu_call_result[start:end], stream=st)
            so[1].copy_to_host(gpu_put_result[start:end],  stream=st)
        # All 3 stages queued for all streams; GPU overlaps them automatically

    # ── Warm-Up (3 passes, fully synchronised) ───────────────────────────────
    print("\n--- GPU Streams Warm Up Started ---")
    for _ in range(3):
        dispatch_all_streams()
        cuda.synchronize()
    print("--- GPU Streams Warm Up Done ---")

    # ── Streamed Benchmark ────────────────────────────────────────────────────
    stream_times = []
    print("\n--- Starting GPU Streams Benchmark ---")
    for i in range(iterations):
        t0 = time.perf_counter()
        dispatch_all_streams()
        cuda.synchronize()          # wait for EVERY stream to finish
        stream_times.append(time.perf_counter() - t0)
        print(f"  Streams iteration {i+1}: {stream_times[-1]:.4f}s")

    avg_stream_time = np.mean(stream_times)
    print(f"Average Streams time (H2D+Kernel+D2H): {avg_stream_time:.4f}s")

    # ── Results & Speedups ────────────────────────────────────────────────────
    stream_speedup = avg_cpu_time / avg_stream_time
    print(f"\n--- Performance Summary ---")
    print(f"GPU Streams is {stream_speedup:.2f}x faster than CPU (full pipeline).")

    # ── Verification ──────────────────────────────────────────────────────────
    print("\n--- Verifying Results ---")
    sample_indices = np.random.choice(num_options, 10, replace=False)
    print("Sample Call — CPU:", cpu_call_result[sample_indices])
    print("Sample Call — GPU:", gpu_call_result[sample_indices])

    call_ok = np.allclose(cpu_call_result, gpu_call_result, rtol=1e-4, atol=1e-6)
    put_ok  = np.allclose(cpu_put_result,  gpu_put_result,  rtol=1e-4, atol=1e-6)
    if call_ok and put_ok:
        print("Results match between CPU and GPU (within tolerance).")
    else:
        print("WARNING: Results do NOT match.")
        print(f"  Max call diff: {np.max(np.abs(cpu_call_result - gpu_call_result)):.2e}")
        print(f"  Max put  diff: {np.max(np.abs(cpu_put_result  - gpu_put_result )):.2e}")


run_benchmark(num_options=(10**8) * 1, iterations=5, num_streams=8)


Benchmarking with 100000000 options, 5 iterations, 8 streams.

--- Starting CPU Benchmark ---
  CPU iteration 1: 12.8255s
  CPU iteration 2: 12.8108s
  CPU iteration 3: 12.9675s
  CPU iteration 4: 12.8233s
  CPU iteration 5: 12.9297s
Average CPU time: 12.8714s

--- GPU Streams Warm Up Started ---
--- GPU Streams Warm Up Done ---

--- Starting GPU Streams Benchmark ---
  Streams iteration 1: 0.3747s
  Streams iteration 2: 0.3740s
  Streams iteration 3: 0.3753s
  Streams iteration 4: 0.3744s
  Streams iteration 5: 0.3744s
Average Streams time (H2D+Kernel+D2H): 0.3745s

--- Performance Summary ---
GPU Streams is 34.37x faster than CPU (full pipeline).

--- Verifying Results ---
Sample Call — CPU: [10.97212405  8.98799198 49.10857824 21.64761811 13.31766179 12.60110805
 25.82739995 29.85409807  2.31476154 43.02943625]
Sample Call — GPU: [10.97212405  8.98799198 49.10857824 21.64761811 13.31766179 12.60110805
 25.82739995 29.85409807  2.31476154 43.02943625]
Results match between CPU and GP

In [1]:
import numpy as np
import time
from numba import cuda
import math

# --- CPU Version (unchanged) ---
def cnd_cpu(d):
    A1 = 0.31938153; A2 = -0.356563782; A3 = 1.781477937
    A4 = -1.821255978; A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S, X, T, R, V = stockPrice, optionStrike, optionYears, Riskfree, Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1); cndd2 = cnd_cpu(d2)
    expRT = np.exp(-R * T)
    return S * cndd1 - X * expRT * cndd2, X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# --- GPU Kernel (unchanged) ---
@cuda.jit(device=True, inline=True, fastmath=True)
def cnd_device(d):
    A1 = 0.31938153; A2 = -0.356563782; A3 = 1.781477937
    A4 = -1.821255978; A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d))
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return (1.0 - ret_val) if d > 0 else ret_val

@cuda.jit(fastmath=True)
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility,
                          callResult, putResult):
    tid = cuda.grid(1)
    N = stockPrice.shape[0]
    if tid < N:
        S, X, T, R, V = stockPrice[tid], optionStrike[tid], optionYears[tid], Riskfree[tid], Volatility[tid]
        sqrtT = math.sqrt(T)
        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
        d2 = d1 - V * sqrtT
        cndd1 = cnd_device(d1); cndd2 = cnd_device(d2)
        expRT = math.exp(-R * T)
        callResult[tid] = S * cndd1 - X * expRT * cndd2
        putResult[tid]  = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)


def run_benchmark(num_options=10**7, iterations=5, num_streams=4):

    np.random.seed(42)
    stockPrice   = np.random.uniform(50,   150,  num_options).astype(np.float64)
    optionStrike = np.random.uniform(50,   150,  num_options).astype(np.float64)
    optionYears  = np.random.uniform(0.5,  2.0,  num_options).astype(np.float64)
    Riskfree     = np.random.uniform(0.01, 0.05, num_options).astype(np.float64)
    Volatility   = np.random.uniform(0.1,  0.5,  num_options).astype(np.float64)

    print(f"Benchmarking with {num_options} options, {iterations} iterations, {num_streams} streams.")

    # ── CPU Benchmark ─────────────────────────────────────────────────────────
    cpu_call_result, cpu_put_result = black_scholes_cpu(
        stockPrice, optionStrike, optionYears, Riskfree, Volatility)  # warm-up

    cpu_times = []
    print("\n--- Starting CPU Benchmark ---")
    for i in range(iterations):
        t0 = time.perf_counter()
        cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility)
        cpu_times.append(time.perf_counter() - t0)
        print(f"  CPU iteration {i+1}: {cpu_times[-1]:.4f}s")
    avg_cpu_time = np.mean(cpu_times)
    print(f"Average CPU time: {avg_cpu_time:.4f}s")

    if not cuda.is_available():
        print("\nCUDA not available. Skipping GPU benchmark.")
        return

    # ── Pinned Memory Allocation ──────────────────────────────────────────────
    # Input arrays: pinned pageable→pinned copy
    pin_stockPrice   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionStrike = cuda.pinned_array(num_options, dtype=np.float64)
    pin_optionYears  = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Riskfree     = cuda.pinned_array(num_options, dtype=np.float64)
    pin_Volatility   = cuda.pinned_array(num_options, dtype=np.float64)
    pin_stockPrice[:]   = stockPrice
    pin_optionStrike[:] = optionStrike
    pin_optionYears[:]  = optionYears
    pin_Riskfree[:]     = Riskfree
    pin_Volatility[:]   = Volatility

    # Output arrays: pinned so D2H is also fast
    gpu_call_result = cuda.pinned_array(num_options, dtype=np.float64)
    gpu_put_result  = cuda.pinned_array(num_options, dtype=np.float64)

    # ── Stream Setup ──────────────────────────────────────────────────────────
    # Divide work evenly; last chunk absorbs any remainder
    chunk_size = (num_options + num_streams - 1) // num_streams

    streams = [cuda.stream() for _ in range(num_streams)]

    # Pre-allocate one set of GPU arrays per stream (reused every iteration)
    # This avoids repeated GPU malloc inside the timed loop
    d_in  = []   # list of [d_S, d_X, d_T, d_R, d_V] per stream
    d_out = []   # list of [d_call, d_put]           per stream

    for s in range(num_streams):
        start = s * chunk_size
        end   = min(start + chunk_size, num_options)
        size  = end - start
        d_in.append([
            cuda.device_array(size, dtype=np.float64),  # stockPrice
            cuda.device_array(size, dtype=np.float64),  # optionStrike
            cuda.device_array(size, dtype=np.float64),  # optionYears
            cuda.device_array(size, dtype=np.float64),  # Riskfree
            cuda.device_array(size, dtype=np.float64),  # Volatility
        ])
        d_out.append([
            cuda.device_array(size, dtype=np.float64),  # callResult
            cuda.device_array(size, dtype=np.float64),  # putResult
        ])

    tpb = 256  # threads per block

    # Helper: dispatch one full pipeline pass across all streams
    def dispatch_all_streams():
        for s in range(num_streams):
            start  = s * chunk_size
            end    = min(start + chunk_size, num_options)
            size   = end - start
            blocks = (size + tpb - 1) // tpb
            st     = streams[s]
            si, so = d_in[s], d_out[s]

            # ① Async H2D: pinned host chunk → GPU (no CPU stall)
            si[0].copy_to_device(pin_stockPrice[start:end],   stream=st)
            si[1].copy_to_device(pin_optionStrike[start:end], stream=st)
            si[2].copy_to_device(pin_optionYears[start:end],  stream=st)
            si[3].copy_to_device(pin_Riskfree[start:end],     stream=st)
            si[4].copy_to_device(pin_Volatility[start:end],   stream=st)

            # ② Async Kernel: runs after H2D completes ON THIS STREAM
            black_scholes_kernel[blocks, tpb, st](
                si[0], si[1], si[2], si[3], si[4], so[0], so[1])

            # ③ Async D2H: runs after kernel completes ON THIS STREAM
            so[0].copy_to_host(gpu_call_result[start:end], stream=st)
            so[1].copy_to_host(gpu_put_result[start:end],  stream=st)
        # All 3 stages queued for all streams; GPU overlaps them automatically

    # ── Warm-Up (3 passes, fully synchronised) ───────────────────────────────
    print("\n--- GPU Streams Warm Up Started ---")
    for _ in range(3):
        dispatch_all_streams()
        cuda.synchronize()
    print("--- GPU Streams Warm Up Done ---")

    # ── Streamed Benchmark ────────────────────────────────────────────────────
    stream_times = []
    print("\n--- Starting GPU Streams Benchmark ---")
    for i in range(iterations):
        t0 = time.perf_counter()
        dispatch_all_streams()
        cuda.synchronize()          # wait for EVERY stream to finish
        stream_times.append(time.perf_counter() - t0)
        print(f"  Streams iteration {i+1}: {stream_times[-1]:.4f}s")

    avg_stream_time = np.mean(stream_times)
    print(f"Average Streams time (H2D+Kernel+D2H): {avg_stream_time:.4f}s")

    # ── Results & Speedups ────────────────────────────────────────────────────
    stream_speedup = avg_cpu_time / avg_stream_time
    print(f"\n--- Performance Summary ---")
    print(f"GPU Streams is {stream_speedup:.2f}x faster than CPU (full pipeline).")

    # ── Verification ──────────────────────────────────────────────────────────
    print("\n--- Verifying Results ---")
    sample_indices = np.random.choice(num_options, 10, replace=False)
    print("Sample Call — CPU:", cpu_call_result[sample_indices])
    print("Sample Call — GPU:", gpu_call_result[sample_indices])

    call_ok = np.allclose(cpu_call_result, gpu_call_result, rtol=1e-4, atol=1e-6)
    put_ok  = np.allclose(cpu_put_result,  gpu_put_result,  rtol=1e-4, atol=1e-6)
    if call_ok and put_ok:
        print("Results match between CPU and GPU (within tolerance).")
    else:
        print("WARNING: Results do NOT match.")
        print(f"  Max call diff: {np.max(np.abs(cpu_call_result - gpu_call_result)):.2e}")
        print(f"  Max put  diff: {np.max(np.abs(cpu_put_result  - gpu_put_result )):.2e}")


run_benchmark(num_options=(10**8) * 2, iterations=5, num_streams=8)


Benchmarking with 200000000 options, 5 iterations, 8 streams.

--- Starting CPU Benchmark ---
  CPU iteration 1: 25.9202s
  CPU iteration 2: 25.8064s
  CPU iteration 3: 25.9018s
  CPU iteration 4: 25.7031s
  CPU iteration 5: 25.7759s
Average CPU time: 25.8215s

--- GPU Streams Warm Up Started ---
--- GPU Streams Warm Up Done ---

--- Starting GPU Streams Benchmark ---
  Streams iteration 1: 0.7488s
  Streams iteration 2: 0.7482s
  Streams iteration 3: 0.7492s
  Streams iteration 4: 0.7478s
  Streams iteration 5: 0.7491s
Average Streams time (H2D+Kernel+D2H): 0.7486s

--- Performance Summary ---
GPU Streams is 34.49x faster than CPU (full pipeline).

--- Verifying Results ---
Sample Call — CPU: [10.0332485  84.86172617  0.8517943  40.12633017  1.49828782 41.38112611
 46.07402779 32.82489325 61.25152044 18.40847933]
Sample Call — GPU: [10.0332485  84.86172617  0.8517943  40.12633017  1.49828782 41.38112611
 46.07402779 32.82489325 61.25152044 18.40847933]
Results match between CPU and GP

## Pageable Memory

In [7]:
import numpy as np
import time
from numba import cuda
import math

# --- CPU Version ---
def cnd_cpu(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S = stockPrice
    X = optionStrike
    T = optionYears
    R = Riskfree
    V = Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1)
    cndd2 = cnd_cpu(d2)

    expRT = np.exp(- R * T)

    callResult = S * cndd1 - X * expRT * cndd2
    putResult = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

    return callResult, putResult

# --- GPU Version ---
@cuda.jit(device=True, inline=True, fastmath=True)
def cnd_device(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d))
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    if d > 0:
        return 1.0 - ret_val
    else:
        return ret_val

@cuda.jit(fastmath=True)
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility, callResult, putResult):
    tid = cuda.grid(1)
    N = stockPrice.shape[0]

    if tid < N:
        S = stockPrice[tid]
        X = optionStrike[tid]
        T = optionYears[tid]
        R = Riskfree[tid]
        V = Volatility[tid]

        sqrtT = math.sqrt(T)
        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
        d2 = d1 - V * sqrtT

        cndd1 = cnd_device(d1)
        cndd2 = cnd_device(d2)

        expRT = math.exp(- R * T)

        callResult[tid] = S * cndd1 - X * expRT * cndd2
        putResult[tid] = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# --- Benchmarking Setup ---
def run_benchmark(num_options=10**7, iterations=5):
    # Generate random input data
    np.random.seed(42) # for reproducibility
    stockPrice = np.random.uniform(50, 150, num_options).astype(np.float64)
    optionStrike = np.random.uniform(50, 150, num_options).astype(np.float64)
    optionYears = np.random.uniform(0.5, 2.0, num_options).astype(np.float64)
    Riskfree = np.random.uniform(0.01, 0.05, num_options).astype(np.float64)
    Volatility = np.random.uniform(0.1, 0.5, num_options).astype(np.float64)

    print(f"Benchmarking with {num_options} options over {iterations} iterations.")

    # --- CPU Benchmark ---
    cpu_times = []
    cpu_call_result = None
    cpu_put_result = None

    print("\n--- CPU Warm Up Started ---")
    # Warm Up CPU
    cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility
        )
    print("\n--- CPU Warm Up Done ---")
    
    print("\n--- Starting CPU Benchmark ---")
    for i in range(iterations):
        start_time = time.perf_counter()
        cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility
        )
        end_time = time.perf_counter()
        cpu_times.append(end_time - start_time)
        print(f"CPU iteration {i+1}: {cpu_times[-1]:.4f} seconds")
    avg_cpu_time = np.mean(cpu_times)
    print(f"Average CPU time: {avg_cpu_time:.4f} seconds")

    # --- GPU Benchmark ---
    if not cuda.is_available():
        print("\nCUDA is not available. Skipping GPU benchmark.")
        return

    gpu_times = []

    # ── WINDOW 1: Host → Device (H2D) transfer only ──────────────────────────
    t0 = time.perf_counter()
    
    # Allocate device memory
    d_stockPrice = cuda.to_device(stockPrice)
    d_optionStrike = cuda.to_device(optionStrike)
    d_optionYears = cuda.to_device(optionYears)
    d_Riskfree = cuda.to_device(Riskfree)
    d_Volatility = cuda.to_device(Volatility)
    d_callResult = cuda.device_array_like(stockPrice)
    d_putResult = cuda.device_array_like(stockPrice)

    cuda.synchronize()               # ← MUST sync; to_device is async
    t1 = time.perf_counter()
    h2d_time = t1 - t0
    print(f"H2D Transfer:    {h2d_time:.4f}s")
    

    # Determine kernel launch configuration
    threadsperblock = 256 # A common choice for good occupancy, adjust based on GPU architecture
    blockspergrid = (num_options + threadsperblock - 1) // threadsperblock
    print(f"GPU Launch Config: {blockspergrid} blocks, {threadsperblock} threads per block")

    
    print("\n--- GPU Warm Up Started ---")
    for i in range(3):
        black_scholes_kernel[blockspergrid, threadsperblock](
                d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
                d_callResult, d_putResult
            )
        cuda.synchronize() # Wait for GPU to finish
    print("\n--- GPU Warm Up Done ---")
    print("\n--- Starting GPU Benchmark ---")

    for i in range(iterations):
        # ── WINDOW 2: Pure kernel time (what you already have) ───────────────────
        start_time = time.perf_counter()
        black_scholes_kernel[blockspergrid, threadsperblock](
            d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
            d_callResult, d_putResult
        )
        cuda.synchronize() # Wait for GPU to finish
        end_time = time.perf_counter()
        gpu_times.append(end_time - start_time)
        print(f"GPU iteration {i+1}: {gpu_times[-1]:.4f} seconds")
        
    avg_gpu_time = np.mean(gpu_times)
    print(f"Average GPU time: {avg_gpu_time:.4f} seconds")

    
    # ── WINDOW 3: Device → Host (D2H) transfer only ──────────────────────────
    t2 = time.perf_counter()
    # Transfer results back to host for verification
    gpu_call_result = np.empty_like(stockPrice)
    gpu_put_result = np.empty_like(stockPrice)
    d_callResult.copy_to_host(gpu_call_result)
    d_putResult.copy_to_host(gpu_put_result)
    cuda.synchronize()
    t3 = time.perf_counter()
    d2h_time = t3 - t2
    print(f"D2H Transfer:    {d2h_time:.4f}s")

    # --- Verification ---
    if cpu_call_result is not None and cpu_put_result is not None:
        print("\n--- Verifying Results ---")
        # Check a small subset for numerical stability differences
        sample_indices = np.random.choice(num_options, min(10, num_options), replace=False)

        print("\nSample Call Results:")
        print("CPU:", cpu_call_result[sample_indices])
        print("GPU:", gpu_call_result[sample_indices])

        print("\nSample Put Results:")
        print("CPU:", cpu_put_result[sample_indices])
        print("GPU:", gpu_put_result[sample_indices])

        # Use allclose for floating point comparison
        # Increased rtol/atol slightly due to fastmath and different math library implementations
        call_diff = np.allclose(cpu_call_result, gpu_call_result, rtol=1e-4, atol=1e-6)
        put_diff = np.allclose(cpu_put_result, gpu_put_result, rtol=1e-4, atol=1e-6)

        if call_diff and put_diff:
            print("Results match between CPU and GPU (within tolerance).")
        else:
            print("WARNING: Results do NOT perfectly match between CPU and GPU.")
            if not call_diff:
                print("Call results differ.")
            if not put_diff:
                print("Put results differ.")
            # Print max absolute difference for debugging
            print(f"Max absolute difference in Call results: {np.max(np.abs(cpu_call_result - gpu_call_result)):.2e}")
            print(f"Max absolute difference in Put results: {np.max(np.abs(cpu_put_result - gpu_put_result)):.2e}")

    if cuda.is_available():
        if avg_gpu_time > 0: # Avoid division by zero if GPU time is somehow 0
            speedup = avg_cpu_time / avg_gpu_time
            print(f"\n--- Performance Summary ---")
            print(f"GPU is {speedup:.2f}x faster than CPU.")
        else:
            print("\nGPU time was zero, cannot calculate speedup.")

# Run the benchmark 
run_benchmark(num_options=(10**8) * 1, iterations=5)

Benchmarking with 100000000 options over 5 iterations.

--- CPU Warm Up Started ---

--- CPU Warm Up Done ---

--- Starting CPU Benchmark ---
CPU iteration 1: 13.2290 seconds
CPU iteration 2: 13.0767 seconds
CPU iteration 3: 13.0742 seconds
CPU iteration 4: 13.0567 seconds
CPU iteration 5: 13.0828 seconds
Average CPU time: 13.1039 seconds
H2D Transfer:    0.8683s
GPU Launch Config: 390625 blocks, 256 threads per block

--- GPU Warm Up Started ---

--- GPU Warm Up Done ---

--- Starting GPU Benchmark ---
GPU iteration 1: 0.1341 seconds
GPU iteration 2: 0.1341 seconds
GPU iteration 3: 0.1341 seconds
GPU iteration 4: 0.1340 seconds
GPU iteration 5: 0.1341 seconds
Average GPU time: 0.1341 seconds
D2H Transfer:    0.5705s

--- Verifying Results ---

Sample Call Results:
CPU: [10.97212405  8.98799198 49.10857824 21.64761811 13.31766179 12.60110805
 25.82739995 29.85409807  2.31476154 43.02943625]
GPU: [10.97212405  8.98799198 49.10857824 21.64761811 13.31766179 12.60110805
 25.82739995 29.85

In [5]:
import numpy as np
import time
from numba import cuda
import math

# --- CPU Version ---
def cnd_cpu(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * np.abs(d))
    ret_val = (RSQRT2PI * np.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    return np.where(d > 0, 1.0 - ret_val, ret_val)

def black_scholes_cpu(stockPrice, optionStrike, optionYears, Riskfree, Volatility):
    S = stockPrice
    X = optionStrike
    T = optionYears
    R = Riskfree
    V = Volatility
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
    d2 = d1 - V * sqrtT
    cndd1 = cnd_cpu(d1)
    cndd2 = cnd_cpu(d2)

    expRT = np.exp(- R * T)

    callResult = S * cndd1 - X * expRT * cndd2
    putResult = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

    return callResult, putResult

# --- GPU Version ---
@cuda.jit(device=True, inline=True, fastmath=True)
def cnd_device(d):
    A1 = 0.31938153
    A2 = -0.356563782
    A3 = 1.781477937
    A4 = -1.821255978
    A5 = 1.330274429
    RSQRT2PI = 0.39894228040143267793994605993438
    K = 1.0 / (1.0 + 0.2316419 * math.fabs(d))
    ret_val = (RSQRT2PI * math.exp(-0.5 * d * d) *
               (K * (A1 + K * (A2 + K * (A3 + K * (A4 + K * A5))))))
    if d > 0:
        return 1.0 - ret_val
    else:
        return ret_val

@cuda.jit(fastmath=True)
def black_scholes_kernel(stockPrice, optionStrike, optionYears, Riskfree, Volatility, callResult, putResult):
    tid = cuda.grid(1)
    N = stockPrice.shape[0]

    if tid < N:
        S = stockPrice[tid]
        X = optionStrike[tid]
        T = optionYears[tid]
        R = Riskfree[tid]
        V = Volatility[tid]

        sqrtT = math.sqrt(T)
        d1 = (math.log(S / X) + (R + 0.5 * V * V) * T) / (V * sqrtT)
        d2 = d1 - V * sqrtT

        cndd1 = cnd_device(d1)
        cndd2 = cnd_device(d2)

        expRT = math.exp(- R * T)

        callResult[tid] = S * cndd1 - X * expRT * cndd2
        putResult[tid] = X * expRT * (1.0 - cndd2) - S * (1.0 - cndd1)

# --- Benchmarking Setup ---
def run_benchmark(num_options=10**7, iterations=5):
    # Generate random input data
    np.random.seed(42) # for reproducibility
    stockPrice = np.random.uniform(50, 150, num_options).astype(np.float64)
    optionStrike = np.random.uniform(50, 150, num_options).astype(np.float64)
    optionYears = np.random.uniform(0.5, 2.0, num_options).astype(np.float64)
    Riskfree = np.random.uniform(0.01, 0.05, num_options).astype(np.float64)
    Volatility = np.random.uniform(0.1, 0.5, num_options).astype(np.float64)

    print(f"Benchmarking with {num_options} options over {iterations} iterations.")

    # --- CPU Benchmark ---
    cpu_times = []
    cpu_call_result = None
    cpu_put_result = None

    print("\n--- CPU Warm Up Started ---")
    # Warm Up CPU
    cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility
        )
    print("\n--- CPU Warm Up Done ---")
    
    print("\n--- Starting CPU Benchmark ---")
    for i in range(iterations):
        start_time = time.perf_counter()
        cpu_call_result, cpu_put_result = black_scholes_cpu(
            stockPrice, optionStrike, optionYears, Riskfree, Volatility
        )
        end_time = time.perf_counter()
        cpu_times.append(end_time - start_time)
        print(f"CPU iteration {i+1}: {cpu_times[-1]:.4f} seconds")
    avg_cpu_time = np.mean(cpu_times)
    print(f"Average CPU time: {avg_cpu_time:.4f} seconds")

    # --- GPU Benchmark ---
    if not cuda.is_available():
        print("\nCUDA is not available. Skipping GPU benchmark.")
        return

    gpu_times = []

    # ── WINDOW 1: Host → Device (H2D) transfer only ──────────────────────────
    t0 = time.perf_counter()
    
    # Allocate device memory
    d_stockPrice = cuda.to_device(stockPrice)
    d_optionStrike = cuda.to_device(optionStrike)
    d_optionYears = cuda.to_device(optionYears)
    d_Riskfree = cuda.to_device(Riskfree)
    d_Volatility = cuda.to_device(Volatility)
    d_callResult = cuda.device_array_like(stockPrice)
    d_putResult = cuda.device_array_like(stockPrice)

    cuda.synchronize()               # ← MUST sync; to_device is async
    t1 = time.perf_counter()
    h2d_time = t1 - t0
    print(f"H2D Transfer:    {h2d_time:.4f}s")
    

    # Determine kernel launch configuration
    threadsperblock = 256 # A common choice for good occupancy, adjust based on GPU architecture
    blockspergrid = (num_options + threadsperblock - 1) // threadsperblock
    print(f"GPU Launch Config: {blockspergrid} blocks, {threadsperblock} threads per block")

    
    print("\n--- GPU Warm Up Started ---")
    for i in range(3):
        black_scholes_kernel[blockspergrid, threadsperblock](
                d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
                d_callResult, d_putResult
            )
        cuda.synchronize() # Wait for GPU to finish
    print("\n--- GPU Warm Up Done ---")
    print("\n--- Starting GPU Benchmark ---")

    for i in range(iterations):
        # ── WINDOW 2: Pure kernel time (what you already have) ───────────────────
        start_time = time.perf_counter()
        black_scholes_kernel[blockspergrid, threadsperblock](
            d_stockPrice, d_optionStrike, d_optionYears, d_Riskfree, d_Volatility,
            d_callResult, d_putResult
        )
        cuda.synchronize() # Wait for GPU to finish
        end_time = time.perf_counter()
        gpu_times.append(end_time - start_time)
        print(f"GPU iteration {i+1}: {gpu_times[-1]:.4f} seconds")
        
    avg_gpu_time = np.mean(gpu_times)
    print(f"Average GPU time: {avg_gpu_time:.4f} seconds")

    
    # ── WINDOW 3: Device → Host (D2H) transfer only ──────────────────────────
    t2 = time.perf_counter()
    # Transfer results back to host for verification
    gpu_call_result = np.empty_like(stockPrice)
    gpu_put_result = np.empty_like(stockPrice)
    d_callResult.copy_to_host(gpu_call_result)
    d_putResult.copy_to_host(gpu_put_result)
    cuda.synchronize()
    t3 = time.perf_counter()
    d2h_time = t3 - t2
    print(f"D2H Transfer:    {d2h_time:.4f}s")

    # --- Verification ---
    if cpu_call_result is not None and cpu_put_result is not None:
        print("\n--- Verifying Results ---")
        # Check a small subset for numerical stability differences
        sample_indices = np.random.choice(num_options, min(10, num_options), replace=False)

        print("\nSample Call Results:")
        print("CPU:", cpu_call_result[sample_indices])
        print("GPU:", gpu_call_result[sample_indices])

        print("\nSample Put Results:")
        print("CPU:", cpu_put_result[sample_indices])
        print("GPU:", gpu_put_result[sample_indices])

        # Use allclose for floating point comparison
        # Increased rtol/atol slightly due to fastmath and different math library implementations
        call_diff = np.allclose(cpu_call_result, gpu_call_result, rtol=1e-4, atol=1e-6)
        put_diff = np.allclose(cpu_put_result, gpu_put_result, rtol=1e-4, atol=1e-6)

        if call_diff and put_diff:
            print("Results match between CPU and GPU (within tolerance).")
        else:
            print("WARNING: Results do NOT perfectly match between CPU and GPU.")
            if not call_diff:
                print("Call results differ.")
            if not put_diff:
                print("Put results differ.")
            # Print max absolute difference for debugging
            print(f"Max absolute difference in Call results: {np.max(np.abs(cpu_call_result - gpu_call_result)):.2e}")
            print(f"Max absolute difference in Put results: {np.max(np.abs(cpu_put_result - gpu_put_result)):.2e}")

    if cuda.is_available():
        if avg_gpu_time > 0: # Avoid division by zero if GPU time is somehow 0
            speedup = avg_cpu_time / avg_gpu_time
            print(f"\n--- Performance Summary ---")
            print(f"GPU is {speedup:.2f}x faster than CPU.")
        else:
            print("\nGPU time was zero, cannot calculate speedup.")

# Run the benchmark 
run_benchmark(num_options=(10**8) * 2, iterations=5)

Benchmarking with 200000000 options over 5 iterations.

--- CPU Warm Up Started ---

--- CPU Warm Up Done ---

--- Starting CPU Benchmark ---
CPU iteration 1: 27.7256 seconds
CPU iteration 2: 26.2623 seconds
CPU iteration 3: 26.2826 seconds
CPU iteration 4: 26.2581 seconds
CPU iteration 5: 26.2045 seconds
Average CPU time: 26.5466 seconds
H2D Transfer:    1.7435s
GPU Launch Config: 781250 blocks, 256 threads per block

--- GPU Warm Up Started ---

--- GPU Warm Up Done ---

--- Starting GPU Benchmark ---
GPU iteration 1: 0.2683 seconds
GPU iteration 2: 0.2686 seconds
GPU iteration 3: 0.2681 seconds
GPU iteration 4: 0.2677 seconds
GPU iteration 5: 0.2677 seconds
Average GPU time: 0.2681 seconds
D2H Transfer:    1.1373s

--- Verifying Results ---

Sample Call Results:
CPU: [10.0332485  84.86172617  0.8517943  40.12633017  1.49828782 41.38112611
 46.07402779 32.82489325 61.25152044 18.40847933]
GPU: [10.0332485  84.86172617  0.8517943  40.12633017  1.49828782 41.38112611
 46.07402779 32.82